In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# =========================
# LOAD DATA
# =========================
train_ds = keras.utils.image_dataset_from_directory(
    r"C:\Users\paula\OneDrive\Documentos\GitHub\Smart-Waste-Classification-NN-PROJECT-\data_processed\train",
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = keras.utils.image_dataset_from_directory(
    r"C:\Users\paula\OneDrive\Documentos\GitHub\Smart-Waste-Classification-NN-PROJECT-\data_processed\val",
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_ds.class_names
np.save("classes.npy", class_names)
num_classes = len(class_names)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)

# =========================
# MODEL (ResNet50)
# =========================

# 1. Swap MobileNetV2 for ResNet50
base_model = keras.applications.ResNet50(
    input_shape=(*IMG_SIZE, 3),
    include_top=False,
    weights="imagenet"
)
base_model.trainable = False

model = keras.Sequential([
    layers.Input(shape=(*IMG_SIZE, 3)),

    # 2. ResNet50 expects different preprocessing than MobileNetV2
    # We use the built-in ResNet50 preprocessing function
    layers.Lambda(tf.keras.applications.resnet50.preprocess_input),

    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),

    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.4), # Slightly higher dropout for a larger model
    layers.Dense(num_classes, activation="softmax")
])

model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=[
        keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True)
    ]
)

model.save("waste_classifier_resnet50.keras")
print("✅ ResNet50 Model saved")

Found 5600 files belonging to 4 classes.
Using 4480 files for training.
Found 300 files belonging to 4 classes.
Using 60 files for validation.
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 27s 0us/step

Epoch 1/15
140/140 ━━━━━━━━━━━━━━━━━━━━ 121s 832ms/step - accuracy: 0.8225 - loss: 0.4839 - val_accuracy: 0.9667 - val_loss: 0.1530
Epoch 2/15
140/140 ━━━━━━━━━━━━━━━━━━━━ 105s 749ms/step - accuracy: 0.9455 - loss: 0.1584 - val_accuracy: 0.9833 - val_loss: 0.0686
Epoch 3/15
140/140 ━━━━━━━━━━━━━━━━━━━━ 103s 738ms/step - accuracy: 0.9652 - loss: 0.1078 - val_accuracy: 0.9833 - val_loss: 0.0568
Epoch 4/15
140/140 ━━━━━━━━━━━━━━━━━━━━ 117s 834ms/step - accuracy: 0.9721 - loss: 0.0872 - val_accuracy: 1.0000 - val_loss: 0.0191
Epoch 5/15
140/140 ━━━━━━━━━━━━━━━━━━━━ 113s 811ms/step - accuracy: 0.9783 - loss: 0.0706 - val_accuracy: 1.0000 - val_loss: 0.0150
Epoch 6/15
140/140 ━━━━━━━━━━━━━━━━━━━━ 116s 828ms/step - accuracy: 0.9848 - loss: 0.0543 - val_accuracy: 1.0000 - val_loss: 0.0136
Epoch 7/15
1

In [3]:
import tensorflow as tf
import numpy as np

# =========================
# LOAD MODEL + CLASSES
# =========================
# We define the custom objects mapping so Keras knows what the Lambda layer is
custom_objects = {"tf": tf} 

model = tf.keras.models.load_model(
    "waste_classifier_resnet50.keras", 
    custom_objects=custom_objects
)
class_names = np.load("classes.npy", allow_pickle=True)
# =========================
# PREDICT FUNCTION
# =========================
def predict_image(img_path):
    # 1. Load the image
    img = tf.keras.utils.load_img(img_path, target_size=IMG_SIZE)
    img_array = tf.keras.utils.img_to_array(img)

    # 2. Add batch dimension: (1, 224, 224, 3)
    img_array = np.expand_dims(img_array, axis=0)

    # 3. Apply ResNet50 specific preprocessing
    # This handles the BGR conversion and mean subtraction
    img_array = tf.keras.applications.resnet50.preprocess_input(img_array)

    # 4. Run Inference
    preds = model.predict(img_array)

    class_index = np.argmax(preds)
    confidence = np.max(preds)

    print("-" * 30)
    print(f"Predicted Class: {class_names[class_index]}")
    print(f"Confidence:      {confidence:.2%}")
    print("-" * 30)


# =========================
# TEST
# =========================
# Replace with your actual test image path
predict_image(r"C:\Users\paula\Downloads\paper==.jpg")

TypeError: <class 'keras.src.models.sequential.Sequential'> could not be deserialized properly. Please ensure that components that are Python object instances (layers, models, etc.) returned by `get_config()` are explicitly deserialized in the model's `from_config()` method.

config={'module': 'keras', 'class_name': 'Sequential', 'config': {'name': 'sequential', 'trainable': True, 'dtype': {'module': 'keras', 'class_name': 'DTypePolicy', 'config': {'name': 'float32'}, 'registered_name': None, 'shared_object_id': 2556839748544}, 'layers': [{'module': 'keras.layers', 'class_name': 'InputLayer', 'config': {'batch_shape': [None, 224, 224, 3], 'dtype': 'float32', 'sparse': False, 'ragged': False, 'name': 'input_layer_1', 'optional': False}, 'registered_name': None}, {'module': 'keras.layers', 'class_name': 'Lambda', 'config': {'name': 'lambda', 'trainable': True, 'dtype': {'module': 'keras', 'class_name': 'DTypePolicy', 'config': {'name': 'float32'}, 'registered_name': None}, 'function': {'